# Sanjavan Ghodasara

**Cluster 2 Stacking Model**

**Group 2** | Cluster 2 (refreshed data): 424 companies, 113 bankrupt (**26.65% rate**)

**Critical note:** cluster 2's bankruptcy rate (26.65%) now *exceeds* the 20% sparsity cap.
That means the model literally cannot predict all 113 bankrupt companies — at most
~84 predictions can be labeled bankrupt while staying under the cap. This fundamentally
limits achievable accuracy. Best possible custom accuracy is ~74% (84/113). The
KNN-based winner from the prior data version overfits training too hard here
(train sparsity > 20% at thresholds where CV accuracy is acceptable), so the base
stack was switched to **RF + GB + LR** for this version.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.feature_selection import mutual_info_classif

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CLUSTER_DIR = os.path.join(DATA_DIR, 'clusters')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
SUBGROUP_MODEL_DIR = os.path.join(MODEL_DIR, 'subgroup_models')
os.makedirs(SUBGROUP_MODEL_DIR, exist_ok=True)

In [2]:
def custom_accuracy(y_true, y_pred):
    """Project metric: acc = TT / (TT + TF) = recall of bankrupt class."""
    tt = ((y_true == 1) & (y_pred == 1)).sum()
    tf = ((y_true == 1) & (y_pred == 0)).sum()
    if tt + tf == 0:
        return 0.0
    return tt / (tt + tf)

def sparsity_check(y_pred, threshold=0.20):
    rate = y_pred.mean()
    return rate < threshold, rate

def evaluate_model(y_true, y_pred, label=''):
    acc = custom_accuracy(y_true, y_pred)
    ok, rate = sparsity_check(y_pred)
    cm = confusion_matrix(y_true, y_pred)
    tt = int(((y_true == 1) & (y_pred == 1)).sum())
    tf = int(((y_true == 1) & (y_pred == 0)).sum())
    print(f'--- {label} ---')
    print(f'TT (bankrupt predicted bankrupt):     {tt}')
    print(f'TF (bankrupt predicted non-bankrupt): {tf}')
    print(f'TT + TF = {tt + tf} (total bankrupt)')
    print(f'Custom Accuracy (TT/(TT+TF)): {acc:.4f}')
    print(f'Sparsity: {rate:.4f} ({"PASS" if ok else "FAIL"})')
    print(f'Confusion Matrix:\n{cm}')
    print(f'Classification Report:\n{classification_report(y_true, y_pred, zero_division=0)}')
    return acc, rate

---
## Load & Inspect Data

In [3]:
df2 = pd.read_csv(os.path.join(CLUSTER_DIR, 'cluster_2.csv'))
y2 = df2['Bankrupt?'].values
X2 = df2.drop(columns=['Bankrupt?'])
feature_names = X2.columns.tolist()

print(f'Cluster 2: {X2.shape[0]} samples, {X2.shape[1]} features')
print(f'Bankrupt: {int(y2.sum())} ({y2.mean():.4f})')
print(f'Non-bankrupt: {int((y2 == 0).sum())}')
print(f'\nBankruptcy rate: {y2.mean()*100:.2f}%  (EXCEEDS 20% sparsity cap by {y2.mean()*100 - 20:.2f} pts)')
print(f'Max theoretical TT under cap: floor(0.20 * {len(y2)}) = {int(0.20 * len(y2))}')
print(f'Max theoretical custom accuracy: {int(0.20 * len(y2))}/{int(y2.sum())} = {int(0.20 * len(y2))/int(y2.sum()):.4f}')

Cluster 2: 424 samples, 95 features
Bankrupt: 113 (0.2665)
Non-bankrupt: 311

Bankruptcy rate: 26.65%  (EXCEEDS 20% sparsity cap by 6.65 pts)
Max theoretical TT under cap: floor(0.20 * 424) = 84
Max theoretical custom accuracy: 84/113 = 0.7434


## Notes on This Revision

The training data was refreshed. Previous cluster_2 had 706 companies at 18.84% bankruptcy
rate (just under the 20% sparsity cap). The new cluster_2 has 424 companies at **26.65%**
bankruptcy rate (over the cap).

**Consequences:**
1. The 20% sparsity cap is binding — at most 84 companies can be labeled bankrupt.
2. Achievable TT is bounded above by ~84, so custom accuracy ≤ ~74.3% even in the best case.
3. **KNN in the base stack was dropped.** The previous winner (RF + GB + KNN) had such strong
   training overfitting via KNN memorization that training sparsity blew past 20% at any
   threshold where CV accuracy was decent. Switching to **RF + GB + LR** gives a much tighter
   CV/training sparsity gap, letting a single threshold satisfy both constraints.
4. Final config: **RF + GB + LR** base stack (all `class_weight='balanced'`), LR meta (C=2.0).

## Feature Selection

Rank all 95 features by mutual information, then greedily select while dropping any
feature whose pairwise |correlation| with an already-selected feature exceeds 0.85.

In [4]:
mi_scores = mutual_info_classif(X2.values, y2, random_state=RANDOM_STATE)
mi_series = pd.Series(mi_scores, index=feature_names).sort_values(ascending=False)

print('Top 15 features by Mutual Information:')
for i, (feat, score) in enumerate(mi_series.head(15).items()):
    print(f'  {i+1:2d}. {feat[:55]:55s}  MI = {score:.4f}')

corr_matrix = X2.corr().abs()
CORR_THRESH = 0.85
selected = []
for feat in mi_series.index:
    if not selected or not any(corr_matrix.loc[feat, s] > CORR_THRESH for s in selected):
        selected.append(feat)
    if len(selected) >= 10:
        break

N_FEATURES = 5
top_features = selected[:N_FEATURES]
X2_sel = df2[top_features].values

print(f'\nSelected {N_FEATURES} diverse features (pairwise |r| < {CORR_THRESH}):')
for i, feat in enumerate(top_features):
    print(f'  {i+1}. {feat[:55]:55s}  MI = {mi_series[feat]:.4f}')

Top 15 features by Mutual Information:
   1. Borrowing dependency                                     MI = 0.0918
   2. Total debt/Total net worth                               MI = 0.0857
   3. Debt ratio %                                             MI = 0.0827
   4. Net worth/Assets                                         MI = 0.0825
   5. Equity to Liability                                      MI = 0.0812
   6. Liability to Equity                                      MI = 0.0798
   7. Net profit before tax/Paid-in capital                    MI = 0.0702
   8. Persistent EPS in the Last Four Seasons                  MI = 0.0700
   9. Net Value Per Share (B)                                  MI = 0.0654
  10. Per Share Net profit before tax (Yuan ¥)                 MI = 0.0572
  11. Net Income to Stockholder's Equity                       MI = 0.0537
  12. Net Value Per Share (A)                                  MI = 0.0536
  13. Total income/Total expense                             

## Stacking Model Definition

**Base models (3 — model-type diversity):**
1. **Random Forest** — bagged trees, `class_weight='balanced'`
2. **Gradient Boosting** — boosted trees, handles imbalance through the loss
3. **Logistic Regression** — linear model, `class_weight='balanced'`, C=1.0 — contributes
   a smooth decision boundary the trees cannot capture

**Meta model:** `LogisticRegression(C=2.0)` — mild regularization, unweighted

**Preprocessing:** `StandardScaler` (per CV fold to avoid leakage) — LR benefits.

In [5]:
BEST_THRESH = 0.45

print(f'Using {N_FEATURES} features, threshold={BEST_THRESH:.2f}')

base_estimators = [
    ('rf', RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05,
        min_samples_leaf=10, random_state=RANDOM_STATE)),
    ('lr', LogisticRegression(
        class_weight='balanced', C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
]

meta_learner = LogisticRegression(
    C=2.0, max_iter=1000, random_state=RANDOM_STATE)

print('Stacking classifier: RF + GB + LR base models + LR(C=2.0) meta-learner.')

Using 5 features, threshold=0.45
Stacking classifier: RF + GB + LR base models + LR(C=2.0) meta-learner.


## Individual Base-Model Diagnostics

Before stacking, print confusion matrices for each base model on its own
(5-fold stratified predictions with per-fold StandardScaler). These are diagnostic
(default threshold) and don't affect the final stacked predictions.

In [6]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def base_cv_predict(model_factory, X_raw, y):
    preds = np.zeros(len(y), dtype=int)
    for train_idx, val_idx in outer_cv.split(X_raw, y):
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_raw[train_idx])
        X_val = sc.transform(X_raw[val_idx])
        m = model_factory()
        m.fit(X_tr, y[train_idx])
        preds[val_idx] = m.predict(X_val)
    return preds

rf_preds_cv = base_cv_predict(
    lambda: RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    X2_sel, y2)
evaluate_model(y2, rf_preds_cv, 'Base: Random Forest (CV preds, thresh=0.5)')

gb_preds_cv = base_cv_predict(
    lambda: GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05,
        min_samples_leaf=10, random_state=RANDOM_STATE),
    X2_sel, y2)
evaluate_model(y2, gb_preds_cv, 'Base: Gradient Boosting (CV preds, thresh=0.5)')

lr_preds_cv = base_cv_predict(
    lambda: LogisticRegression(class_weight='balanced', C=1.0, max_iter=2000,
                               random_state=RANDOM_STATE),
    X2_sel, y2)
evaluate_model(y2, lr_preds_cv, 'Base: Logistic Regression (CV preds, thresh=0.5)')

--- Base: Random Forest (CV preds, thresh=0.5) ---
TT (bankrupt predicted bankrupt):     67
TF (bankrupt predicted non-bankrupt): 46
TT + TF = 113 (total bankrupt)
Custom Accuracy (TT/(TT+TF)): 0.5929
Sparsity: 0.3373 (FAIL)
Confusion Matrix:
[[235  76]
 [ 46  67]]
Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.76      0.79       311
           1       0.47      0.59      0.52       113

    accuracy                           0.71       424
   macro avg       0.65      0.67      0.66       424
weighted avg       0.74      0.71      0.72       424



--- Base: Gradient Boosting (CV preds, thresh=0.5) ---
TT (bankrupt predicted bankrupt):     44
TF (bankrupt predicted non-bankrupt): 69
TT + TF = 113 (total bankrupt)
Custom Accuracy (TT/(TT+TF)): 0.3894
Sparsity: 0.1769 (PASS)
Confusion Matrix:
[[280  31]
 [ 69  44]]
Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.90      0.85       311
           1       0.59      0.39      0.47       113

    accuracy                           0.76       424
   macro avg       0.69      0.64      0.66       424
weighted avg       0.74      0.76      0.75       424

--- Base: Logistic Regression (CV preds, thresh=0.5) ---
TT (bankrupt predicted bankrupt):     69
TF (bankrupt predicted non-bankrupt): 44
TT + TF = 113 (total bankrupt)
Custom Accuracy (TT/(TT+TF)): 0.6106
Sparsity: 0.3491 (FAIL)
Confusion Matrix:
[[232  79]
 [ 44  69]]
Classification Report:
              precision    recall  f1-score   support

           0       0.84      0

(np.float64(0.6106194690265486), np.float64(0.3490566037735849))

## Cross-Validation (Stacked Model)

5-fold stratified CV. StandardScaler fit per training fold and applied to the
corresponding validation fold — no scaling leakage.

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y2_cv_probas = np.zeros(len(y2))

for fold, (train_idx, val_idx) in enumerate(cv.split(X2_sel, y2)):
    sc_fold = StandardScaler()
    X_train_cv = sc_fold.fit_transform(X2_sel[train_idx])
    X_val_cv = sc_fold.transform(X2_sel[val_idx])
    y_train_cv, y_val_cv = y2[train_idx], y2[val_idx]

    stacker_fold = StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(
                n_estimators=200, max_depth=6, min_samples_leaf=5,
                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
            ('gb', GradientBoostingClassifier(
                n_estimators=150, max_depth=3, learning_rate=0.05,
                min_samples_leaf=10, random_state=RANDOM_STATE)),
            ('lr', LogisticRegression(
                class_weight='balanced', C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
        ],
        final_estimator=LogisticRegression(
            C=2.0, max_iter=1000, random_state=RANDOM_STATE),
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        stack_method='predict_proba',
        n_jobs=-1
    )
    stacker_fold.fit(X_train_cv, y_train_cv)
    y2_cv_probas[val_idx] = stacker_fold.predict_proba(X_val_cv)[:, 1]
    print(f'Fold {fold+1}: val positives={int(y_val_cv.sum())}, avg proba bankrupt={y2_cv_probas[val_idx].mean():.4f}')

print('\n--- Threshold Sweep (CV) ---')
best_thresh, best_acc = 0.5, 0.0
for thresh in np.arange(0.05, 0.96, 0.01):
    preds = (y2_cv_probas >= thresh).astype(int)
    acc = custom_accuracy(y2, preds)
    _, rate = sparsity_check(preds)
    if rate < 0.20 and acc >= best_acc:
        best_acc = acc
        best_thresh = thresh
    if round(thresh, 2) in [0.20, 0.30, 0.35, 0.40, 0.42, 0.44, 0.45, 0.46, 0.48, 0.50, 0.55, 0.60]:
        tt = int(((y2 == 1) & (preds == 1)).sum())
        tf = int(((y2 == 1) & (preds == 0)).sum())
        print(f'  thresh={thresh:.2f}: TT={tt:3d}, TF={tf:3d}, acc={acc:.4f}, sparsity={rate:.4f} {"PASS" if rate < 0.20 else "FAIL"}')

print(f'\nSelected threshold: {BEST_THRESH:.2f} (chosen so both CV and training sparsity pass)')
y2_cv_preds = (y2_cv_probas >= BEST_THRESH).astype(int)
evaluate_model(y2, y2_cv_preds, f'Stacked: Cluster 2 — 5-Fold CV (thresh={BEST_THRESH:.2f})')

Fold 1: val positives=22, avg proba bankrupt=0.2383


Fold 2: val positives=23, avg proba bankrupt=0.2390


Fold 3: val positives=23, avg proba bankrupt=0.2910


Fold 4: val positives=23, avg proba bankrupt=0.2842


Fold 5: val positives=22, avg proba bankrupt=0.2835

--- Threshold Sweep (CV) ---
  thresh=0.20: TT= 92, TF= 21, acc=0.8142, sparsity=0.5495 FAIL
  thresh=0.30: TT= 68, TF= 45, acc=0.6018, sparsity=0.3467 FAIL
  thresh=0.35: TT= 60, TF= 53, acc=0.5310, sparsity=0.2642 FAIL
  thresh=0.40: TT= 50, TF= 63, acc=0.4425, sparsity=0.2075 FAIL
  thresh=0.42: TT= 45, TF= 68, acc=0.3982, sparsity=0.1887 PASS
  thresh=0.44: TT= 41, TF= 72, acc=0.3628, sparsity=0.1722 PASS
  thresh=0.45: TT= 39, TF= 74, acc=0.3451, sparsity=0.1627 PASS
  thresh=0.46: TT= 35, TF= 78, acc=0.3097, sparsity=0.1415 PASS
  thresh=0.48: TT= 31, TF= 82, acc=0.2743, sparsity=0.1179 PASS
  thresh=0.50: TT= 28, TF= 85, acc=0.2478, sparsity=0.1061 PASS
  thresh=0.55: TT= 19, TF= 94, acc=0.1681, sparsity=0.0566 PASS
  thresh=0.60: TT= 13, TF=100, acc=0.1150, sparsity=0.0354 PASS

Selected threshold: 0.45 (chosen so both CV and training sparsity pass)
--- Stacked: Cluster 2 — 5-Fold CV (thresh=0.45) ---
TT (bankrupt predicted b

(np.float64(0.34513274336283184), np.float64(0.16273584905660377))

## Train Final Model & Save

Fit the scaler on the full cluster, fit the stacking classifier, then predict on the
full cluster_2.csv (all 424 rows). These are the Table 3 numbers.

In [8]:
scaler_final = StandardScaler()
X2_scaled = scaler_final.fit_transform(X2_sel)

stacker_final = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=200, max_depth=6, min_samples_leaf=5,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ('gb', GradientBoostingClassifier(
            n_estimators=150, max_depth=3, learning_rate=0.05,
            min_samples_leaf=10, random_state=RANDOM_STATE)),
        ('lr', LogisticRegression(
            class_weight='balanced', C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
    ],
    final_estimator=LogisticRegression(
        C=2.0, max_iter=1000, random_state=RANDOM_STATE),
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    stack_method='predict_proba',
    n_jobs=-1
)
stacker_final.fit(X2_scaled, y2)

y2_train_proba = stacker_final.predict_proba(X2_scaled)[:, 1]
y2_train_pred = (y2_train_proba >= BEST_THRESH).astype(int)
evaluate_model(y2, y2_train_pred, f'Stacked: Cluster 2 — Training on full cluster (thresh={BEST_THRESH:.2f})')

print('\n--- Individual Base Models (trained, full data, thresh=0.5) ---')
for name, est in stacker_final.named_estimators_.items():
    base_preds = est.predict(X2_scaled)
    evaluate_model(y2, base_preds, f'Base(trained): {name}')

--- Stacked: Cluster 2 — Training on full cluster (thresh=0.45) ---
TT (bankrupt predicted bankrupt):     74
TF (bankrupt predicted non-bankrupt): 39
TT + TF = 113 (total bankrupt)
Custom Accuracy (TT/(TT+TF)): 0.6549
Sparsity: 0.1958 (PASS)
Confusion Matrix:
[[302   9]
 [ 39  74]]
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.97      0.93       311
           1       0.89      0.65      0.76       113

    accuracy                           0.89       424
   macro avg       0.89      0.81      0.84       424
weighted avg       0.89      0.89      0.88       424


--- Individual Base Models (trained, full data, thresh=0.5) ---
--- Base(trained): rf ---
TT (bankrupt predicted bankrupt):     97
TF (bankrupt predicted non-bankrupt): 16
TT + TF = 113 (total bankrupt)
Custom Accuracy (TT/(TT+TF)): 0.8584
Sparsity: 0.3467 (FAIL)
Confusion Matrix:
[[261  50]
 [ 16  97]]
Classification Report:
              precision    recall  f1-

In [9]:
model_info = {
    'model': stacker_final,
    'scaler': scaler_final,
    'features': top_features,
    'n_features': N_FEATURES,
    'threshold': BEST_THRESH,
    'cluster_id': 2,
    'model_type': 'stacking',
    'base_models': 'RF + GB + LR',
    'meta_model': 'LogisticRegression(C=2.0)',
    'member': 'Sanjavan Ghodasara',
    'data_version': 'refreshed (424 rows, 26.65% bankrupt)',
}
joblib.dump(model_info, os.path.join(SUBGROUP_MODEL_DIR, 'cluster_2_model.joblib'))
print(f'Cluster 2 model saved. Features: {N_FEATURES}, Threshold: {BEST_THRESH:.2f}')
print(f'Saved keys: {sorted(model_info.keys())}')

Cluster 2 model saved. Features: 5, Threshold: 0.45
Saved keys: ['base_models', 'cluster_id', 'data_version', 'features', 'member', 'meta_model', 'model', 'model_type', 'n_features', 'scaler', 'threshold']


## Summary Cluster 2 (refreshed data)

| Metric | CV (5-fold) | Training |
|---|---|---|
| TT (bankrupt predicted bankrupt) | **39** | **74** |
| TF (bankrupt predicted non-bankrupt) | **74** | **39** |
| TT + TF | 113 | 113 |
| Custom Accuracy (TT/(TT+TF)) | **34.51%** | **65.49%** |
| Sparsity (% predicted bankrupt) | 16.27% **PASS** | 19.58% **PASS** |
| Features (N) | 5 | 5 |
| Threshold | 0.45 | 0.45 |
| Feature Score ((50-N)/50) | 0.90 | 0.90 |

**Estimated Rank Score:** 0.3(0.6549) + 0.4(0.3451) + 0.3(0.90) = **0.6045**

**Top 5 Features (new data, MI + correlation de-dup):**
1. Borrowing dependency
2. Total debt/Total net worth
3. Debt ratio %
4. Equity to Liability
5. Net profit before tax/Paid-in capital

**Final Model:** Stacking (RF + GB + LR → LR C=2.0 meta), all base models `class_weight='balanced'`, no SMOTE, 5-fold CV, per-fold StandardScaler.

### Why the config changed from the old data version

- **Old data (706 rows, 18.84% bankrupt):** RF + GB + KNN won. KNN's local-density signal
  was valuable and the CV/training sparsity gap was manageable because bankruptcy rate
  was under the cap.
- **New data (424 rows, 26.65% bankrupt):** bankruptcy rate exceeds the 20% cap. KNN
  over-commits to minority predictions on training (memorization), which pushes training
  sparsity past 20% at the threshold where CV accuracy is reasonable. Swapping KNN → LR
  gives a much smaller CV/train sparsity gap, allowing a single threshold to satisfy both.

### Saved Artifacts
- `models/subgroup_models/cluster_2_model.joblib` — fitted stack + scaler + feature list + threshold=0.45